# **Sesión: Ejemplo de Agentes para Análisis de datos con CrewAI**

**Objetivo:** Utilizar agentes para analizar CSVs y generar reportes.

---

## **Estructura de la clase**

---

### **1. Introducción a los Agentes (30 minutos)**

#### **¿Qué es un agente?**

1. Tiene un **objetivo**
2. Tiene acceso a **herramientas**
3. Puede ejecutar y planear de manera **autónoma**

![Alt Text](assets/agents_explanation.png)

---

#### **Modelo ReAct**
- **Thought:**
    - Inicia secuencia de 'razonamiento'
    - ¿Qué es lo que necesito hacer?

- **Action:**
    - Decide en una acción -> tiene herramientas/_tools_ disponibles

- **Observation:**
    - Ejecuta su acción con las _tools_ disponibles
    - 'Reflexiona' sobre el resultado de su acción

![Alt Text](assets/react_agents.webp)

---

#### **¿Cuándo usar Agentes?**

1. NO todas las tareas requieren Agentes - KISS (Keep It Simple, Stupid)
2. ¿Requiere capacidades de razonamiento o decisiones en tiempo real? -> Agents, maybe
3. ¿Requieres trabajar con datos no estructurados (imágenes, PDFs, etc.)? -> Agents

---

#### **¿Cómo diseñar Agentes?**

1. ¿Qué quieres lograr?
2. ¿A quiénes necesitas para lograrlo?
3. ¿Qué herramientas les vas a dar para que ejecuten su tarea?
4. Sé lo más **modular** posible

![Alt Text](assets/agent_manager.png)

---


#### **NOTA IMPORTANTE:**

- Hay **MUCHOS** frameworks/maneras de crear agentes, hoy usaremos una de las más sencillas, `crewAI`.

---



### **2. Diseñar un Agente para Análisis de Stocks (30 minutos)**

#### **Caso:**

- Eres un exitoso day trader con un apetito de riesgo medio y quieres invertir $100,000 USD en una acción. 

- Te interesa `AAPL` pero aún no haces la investigación suficiente de las estrategias que más te convienen. 

- Quieres crear un equipo de AI agents que te ahorren el trabajo de investigar cómo le ha ido a `AAPL` en los últimos meses. 

#### **Discutir:** ¿Cómo divides las tareas?, ¿a quiénes necesitas?

---

### **3. Implementación de un Agente para Análisis de Datos con crewAI (40 minutos)**

In [1]:
import pandas as pd
from crewai import Crew, Task, Agent
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from dotenv import load_dotenv
import matplotlib.pyplot as plt

load_dotenv()

True

### **Tools**

- `ScrapeWebsiteTool`: Herramienta que te permite 'scrapear' un website, leer el contenido HTML.
- `SerperDevTool`: Herramienta de búsqueda en Google Search.

In [3]:
scraper_tool = ScrapeWebsiteTool()
online_search_tool = SerperDevTool()

### **Agents**

**Analista de Datos**

In [4]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Estratega de trading**

In [15]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Trade Advisor**

In [5]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Administrador de Riesgos**

In [6]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

### **Tasks**

- A cada agente le vamos a asignar tareas de acuerdo a su **objetivo**. 

In [19]:
from datetime import date

today = date.today().strftime(format='%Y-%m-%d')

financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'start_date': '2025-01-01',
    'today': today,
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

In [7]:
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}) "
        "starting from {start_date} to {today}. "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection} "
        "from {start_date} to {today}."
    ),
    agent=data_analyst_agent,
)

In [16]:
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference}) "
        "and initial capital of {initial_capital}."
        "Consider this timeframe for your analysis tarting from {start_date} to {today}."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [10]:
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
        "Analyze your strategies according to this timeframe: starting from {start_date} to {today}."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [11]:
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
        "Evaluate the risks starting from {start_date} to {today}."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

### **Crew**

#### NOTA:

Los agentes van a ejecutar sus tareas de manera secuencial. 

A menos que les instruyamos otra manera. 

In [17]:
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    verbose=True
)

In [20]:
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

# Agent: Data Analyst
## Task: Continuously monitor and analyze market data for the selected stock (AAPL) starting from 2025-01-01 to 2025-02-24. Use statistical modeling and machine learning to identify trends and predict market movements.


# Agent: Data Analyst
## Thought: I need to gather real-time market data for AAPL from January 1, 2025, to February 24, 2025. Since I can't access real-time machine learning tools directly, my first step will be to find a source that may provide historical data or forecasts for AAPL covering the specified period.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AAPL stock price data historical January 1 2025 to February 24 2025\"}"
## Tool Output: 
{'searchParameters': {'q': 'AAPL stock price data historical January 1 2025 to February 24 2025', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', 'link': 'https://finance.yahoo.com/quote/A

In [22]:
result

CrewOutput(raw='**A Comprehensive Risk Analysis Report for Proposed AAPL Trading Strategies (2025-01-01 to 2025-02-24)**\n\n**1. Strategy 1: Moving Average Crossover**\n\n*Potential Risks:*\n- **Lagging Indicator Risk:** The moving average crossover strategy relies on lagging indicators, making it potentially slow to react to rapid market changes, and thus susceptible to false signals in volatile market conditions. With the data showing AAPL’s closing price swinging from $223.66 to $247.10 across 2025, false positives could lead to significant losses.\n- **Short-term Fluctuations:** The strategy could be affected by daily market noises and fluctuations that do not signify real trend reversals but could trigger trades.\n- **Volatility and Market Sentiments:** Unexpected geopolitical events or economic data releases during this period can cause spikes in volatility, leading to unpredictable crossovers.\n\n*Mitigation Recommendations:*\n- **Enhanced Risk Management Protocols:** Implement 

In [21]:
from IPython.display import Markdown
Markdown(result.raw)

**A Comprehensive Risk Analysis Report for Proposed AAPL Trading Strategies (2025-01-01 to 2025-02-24)**

**1. Strategy 1: Moving Average Crossover**

*Potential Risks:*
- **Lagging Indicator Risk:** The moving average crossover strategy relies on lagging indicators, making it potentially slow to react to rapid market changes, and thus susceptible to false signals in volatile market conditions. With the data showing AAPL’s closing price swinging from $223.66 to $247.10 across 2025, false positives could lead to significant losses.
- **Short-term Fluctuations:** The strategy could be affected by daily market noises and fluctuations that do not signify real trend reversals but could trigger trades.
- **Volatility and Market Sentiments:** Unexpected geopolitical events or economic data releases during this period can cause spikes in volatility, leading to unpredictable crossovers.

*Mitigation Recommendations:*
- **Enhanced Risk Management Protocols:** Implement dynamic stop-loss orders and trailing stops to protect against rapid adverse movements in prices.
- **Supplementary Signals:** Consider using additional indicators such as MACD or Bollinger Bands for more reliable signals alongside moving averages.
- **Position Sizing:** Keep individual trades within 2-5% of the total capital, ensuring medium risk management approach and avoiding overexposure to a single trade.

**2. Strategy 2: Relative Strength Index (RSI) Strategy**

*Potential Risks:*
- **Over-reliance on Single Indicator:** The RSI strategy focuses on overbought/oversold conditions, which can lead to misinterpretations in trending markets as AAPL approaches important events (such as earnings reports noted around late January).
- **Short-term Market Surges:** RSI might not adapt quickly to sudden market news or events that could cause sharp price movements, leading to high-risk entries.
- **Whipsaws in Range-bound Markets:** If AAPL trades within a tight range, RSI signals could lead to multiple false entry/exit signals.

*Mitigation Recommendations:*
- **Broader Analysis Framework:** Use the RSI in conjunction with other technical indicators and fundamental analysis to confirm signals.
- **Profit Objectives and Partial Exits:** Fix profit targets for partial exits to lock in gains while maintaining trend-following positions for larger moves.
- **Regular Strategy Reviews:** Regularly review and adjust RSI thresholds based on evolving market conditions and historical RSI performance relative to its influence on AAPL prices.

**Additional Considerations:**

1. **Market News and Trends:** Stay informed of broader market developments, and potential impacting news on AAPL (e.g., product releases, supply chain news). Employ a flexible strategy to downsize positions post-news events where high volatility is anticipated.
2. **Post-strategy Performance Evaluation:** Regularly backtest strategies with latest market data to tune parameters towards higher efficacy and align with firm’s overall risk policy and profitability thresholds.

By closely monitoring these aspects and adjusting strategies accordingly, risks associated with executing AAPL trades during this specified period can be effectively managed while supporting strategic goals.

In [23]:
financial_trading_crew.kickoff(inputs=financial_trading_inputs)

# Agent: Data Analyst
## Task: Continuously monitor and analyze market data for the selected stock (AAPL) starting from 2025-01-01 to 2025-02-24. Use statistical modeling and machine learning to identify trends and predict market movements.


# Agent: Data Analyst
## Thought: Since I need real-time or recent historical market data for AAPL from January 1, 2025, to February 24, 2025, I should start by searching the internet for any available market data or analysis for AAPL during that period.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AAPL stock market data January 1 2025 to February 24 2025\"}"
## Tool Output: 
{'searchParameters': {'q': 'AAPL stock market data January 1 2025 to February 24 2025', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', 'link': 'https://finance.yahoo.com/quote/AAPL/history/', 'snippet': 'Discover historical prices for AAPL stock on Yahoo Fi

CrewOutput(raw="**Comprehensive Risk Analysis Report for Proposed AAPL Trading Strategies (Jan 1, 2025 - Feb 24, 2025)**\n\n**Summary of Strategies:**\n1. **Breakout Strategy**: Focused on exploiting price breakthroughs beyond set resistance or support levels.\n2. **Mean Reversion Strategy**: Designed to capitalize on short-term price reversals.\n3. **Volume-Based Strategy**: Aims to utilize volume indicators for anticipating significant price movements.\n4. **Momentum Strategy**: Primarily captures profits from sustained price trends.\n\n**Macro and Technical Forecasts:**\n- Predicted AAPL price for early 2025 ranges between $236 to $276, with a possibility of reaching higher levels by mid-year.\n- Despite short-term corrections, consistent upward momentum and expansion in product offerings hint towards a potential rally in 2025, reinforcing a strong buy consensus.\n  \n**Potential Risks and Market Considerations:**\n\n1. **Market Volatility**:\n   - High price fluctuations expected d

In [15]:
print(f'Usamos estos tokens: {result.token_usage}')

Usamos estos tokens: total_tokens=78103 prompt_tokens=72828 cached_prompt_tokens=12160 completion_tokens=5275 successful_requests=13
